In [7]:
#Problem 11:
import numpy as np

# (a) Create arrays and print shapes/sums
I = np.ones((20, 20), dtype=float)
print("I:", I.shape, I.sum())
F = np.ones((10, 10), dtype=float)
print("F:", F.shape, F.sum())

I: (20, 20) 400.0
F: (10, 10) 100.0


In [3]:

# (b) Single-filter, single-map with stride
def conOne(X, W, s=1):
    H, Wd = X.shape
    fh, fw = W.shape
    oh = (H - fh) // s + 1
    ow = (Wd - fw) // s + 1
    Y = np.zeros((oh, ow), dtype=float)
    for i in range(oh):
        for j in range(ow):
            xi = i * s
            yj = j * s
            Y[i, j] = np.sum(X[xi:xi+fh, yj:yj+fw] * W)
    return Y

F_b = np.full((10, 10), 0.01, dtype=float)
Y_b = conOne(I, F_b, s=2)
print("Y (b):", Y_b.shape)

Y (b): (6, 6)


In [4]:

# (c) Demonstrate shapes
x1 = np.ones((2, 2, 3), dtype=float)
print("x1:", x1.shape)
x2 = np.ones((3, 2, 2), dtype=float)
print("x2:", x2.shape)

x1: (2, 2, 3)
x2: (3, 2, 2)


In [5]:

# (d) Many filters -> many feature maps (stacked)
def conTwo(X, Wstack, s=1):
    # X: (H, W), Wstack: (K, fh, fw)
    K, fh, fw = Wstack.shape
    H, Wd = X.shape
    oh = (H - fh) // s + 1
    ow = (Wd - fw) // s + 1
    Y = np.zeros((K, oh, ow), dtype=float)
    for k in range(K):
        Y[k] = conOne(X, Wstack[k], s)
    return Y

F_d = np.stack([
    np.full((10, 10), 0.01, dtype=float),
    np.full((10, 10), 0.01, dtype=float),
    np.full((10, 10), 0.02, dtype=float),
    np.full((10, 10), 0.02, dtype=float)
], axis=0)
Y_d = conTwo(I, F_d, s=2)
print("Y (d):", Y_d.shape)  # expected (4, 6, 6)

Y (d): (4, 6, 6)


In [6]:
# (e) Many input maps -> many output maps (sum over channels)
def conThree(Xstack, Wstack, s=1):
    # Xstack: (C, H, W), Wstack: (K, fh, fw)
    C, H, Wd = Xstack.shape
    K, fh, fw = Wstack.shape
    oh = (H - fh) // s + 1
    ow = (Wd - fw) // s + 1
    Y = np.zeros((K, oh, ow), dtype=float)
    for k in range(K):
        acc = np.zeros((oh, ow), dtype=float)
        for c in range(C):
            acc += conOne(Xstack[c], Wstack[k], s)
        Y[k] = acc
    return Y

# First stage (treat previous output as 1-channel for demonstration)
I3 = np.ones((1, 20, 20), dtype=float)
F1 = np.stack([
    np.full((10, 10), 0.01, dtype=float),
    np.full((10, 10), 0.01, dtype=float),
    np.full((10, 10), 0.02, dtype=float),
    np.full((10, 10), 0.02, dtype=float)
], axis=0)
Y1 = conThree(I3, F1, s=2)  # (4, 6, 6)

# Second stage: 3 filters of size 4x4, stride 1
F2 = np.stack([
    np.full((4, 4), 1/36, dtype=float),
    np.full((4, 4), 1/18, dtype=float),
    np.full((4, 4), 1/9,  dtype=float)
], axis=0)

# Convolve each of the 4 channels of Y1 with each filter in F2 and sum over channels
# Reshape Y1 as input-channel stack
Y1_stack = Y1  # (C=4, H=6, W=6)
Y2 = conThree(Y1_stack, F2, s=1)  # (3, 3, 3)
print("Y1 (e):", Y1_stack.shape, "Y2 (e):", Y2.shape)

Y1 (e): (4, 6, 6) Y2 (e): (3, 3, 3)
